# 12 — SD3 SAE fidelity rescue

Train and compare higher-capacity TopK and timestep-conditioned SAEs on cached SD3 residual activations. Split by original capture sample, never by spatial tokens. Training, validation and output artifacts are kept separate from notebooks 09–11.

In [ ]:
MODE = "pilot"  # "pilot" or "full"
SEED = 2601
EPOCHS = 3 if MODE == "pilot" else 6
MAX_TOKENS_PER_FILE = 384 if MODE == "pilot" else 1024
BATCH = 256 if MODE == "pilot" else 512
RUN_TRAINING = True
CANDIDATES = [
    {"name":"topk_x2_k16","expansion_factor":2,"k":16,"architecture":"topk","steps":[13]},
    {"name":"topk_x4_k32","expansion_factor":4,"k":32,"architecture":"topk","steps":[13]},
    {"name":"topk_x8_k64","expansion_factor":8,"k":64,"architecture":"topk","steps":[13]},
    {"name":"temporal_x4_k32","expansion_factor":4,"k":32,"architecture":"temporal","steps":[0,13,27]},
]
if MODE == "pilot": CANDIDATES = [CANDIDATES[0], CANDIDATES[1], CANDIDATES[3]]

## 1. Environment, reproducible train/validation split

Temporal candidate uses recorded steps 0, 13 and 27. Anchor-only candidates use step 13. All candidate models use the same held-out sample IDs.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display

def root():
    wd=Path.cwd().resolve()
    for p in [wd,*wd.parents,Path("/home/jupyter/project/diffusion_attention_analysis_2")]:
        if (p/"src/diffusion_attention_analysis_v2").is_dir(): return p
    raise FileNotFoundError("Set working directory to the repository root")

ROOT=root(); os.chdir(ROOT); sys.path.insert(0,str(ROOT/"src"))
from diffusion_attention_analysis_v2.sd3_notebook_utils import prepare_datashere_storage
from diffusion_attention_analysis_v2.sae_fidelity_v4 import (
    capture_files, group_file_split, load_research_sae, reconstruction_metrics,
    save_candidate, stable_split_hash, train_candidate)
storage=prepare_datashere_storage(project_root=ROOT,run_tag="sd3_medium_sae_circuit_150",container_name="container4",require_container=True,min_free_gb=12.)
assert storage["free_space_ok"], "Insufficient DataSphere container4 space"
CAP=ROOT/"outputs/sd3_medium_sae_circuit_150/01_capture/captures"
OUT=ROOT/"outputs/sd3_medium_sae_circuit_150/07_sae_fidelity_rescue"
OUT.mkdir(parents=True,exist_ok=True)
assert CAP.is_dir(), f"Missing Stage 01 captures: {CAP}"
assert torch.cuda.is_available(), "Training requires CUDA"
SPLITS={}
for spec in CANDIDATES:
    files=capture_files(CAP,steps=spec["steps"])
    train,val=group_file_split(files,val_fraction=.20,seed=SEED)
    SPLITS[spec["name"]]=(train,val)
    print(spec["name"],"train files",len(train),"validation files",len(val),"sample groups",len({p.parent.parent.parent.name for p in files}))

## 2. Train and evaluate candidates

Checkpoints include architecture metadata and a validation split hash; the temporal encoder is conditioned by continuous diffusion time. Metrics cover normalized RMSE, cosine, explained variance proxy and dead-feature utilization.

In [ ]:
rows=[]
for spec in CANDIDATES:
    name=spec["name"]; train,val=SPLITS[name]; directory=OUT/name
    checkpoint=directory/"sae.pt"; history_path=directory/"training_history.json"
    if checkpoint.exists() and not RUN_TRAINING:
        model,meta=load_research_sae(checkpoint,"cuda")
        history=json.loads(history_path.read_text()) if history_path.exists() else []
    else:
        model,history=train_candidate(train,val,expansion_factor=spec["expansion_factor"],
            k=spec["k"],architecture=spec["architecture"],epochs=EPOCHS,
            batch_size=BATCH,max_tokens=MAX_TOKENS_PER_FILE,device="cuda",seed=SEED)
        metrics=reconstruction_metrics(model,val,batch_size=BATCH,max_tokens=MAX_TOKENS_PER_FILE)
        save_candidate(checkpoint,model,expansion_factor=spec["expansion_factor"],
            k=spec["k"],architecture=spec["architecture"],val_metrics=metrics,
            split_hash=stable_split_hash(val))
        history_path.parent.mkdir(parents=True,exist_ok=True)
        history_path.write_text(json.dumps(history,indent=2))
    val_metrics=reconstruction_metrics(model,val,batch_size=BATCH,max_tokens=MAX_TOKENS_PER_FILE)
    rows.append({"model":name,"checkpoint":str(checkpoint),"architecture":spec["architecture"],
                 "expansion_factor":spec["expansion_factor"],"k":spec["k"],
                 "validation_steps":str(spec["steps"]),"validation_hash":stable_split_hash(val),
                 **val_metrics})
    del model
    torch.cuda.empty_cache()
table=pd.DataFrame(rows).sort_values(["nrmse","cosine"],ascending=[True,False])
table.to_csv(OUT/"candidate_metrics.csv",index=False)
(OUT/"checkpoint_index.json").write_text(json.dumps({r["model"]:r["checkpoint"] for r in rows},indent=2))
display(table)

## 3. Historical baseline and figures

Comparison uses the same anchor-step validation sample split. Validation reconstruction is a necessary diagnostic, not an estimate of causal mediation.

In [ ]:
old=ROOT/"outputs/sd3_medium_sae_circuit_150/03_sae/mid_block11_step13/sae.pt"
if old.is_file():
    val_anchor=[p for p in SPLITS[CANDIDATES[0]["name"]][1] if p.parent.name=="step_013"]
    model,meta=load_research_sae(old,"cuda")
    baseline=reconstruction_metrics(model,val_anchor,batch_size=BATCH,max_tokens=MAX_TOKENS_PER_FILE)
    print("Historical mid SAE on identical held-out anchor split:")
    display(pd.DataFrame([{"model":"historical_mid_topk",**baseline}]))
    del model;torch.cuda.empty_cache()
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(table.model,table.nrmse)
ax.set_ylabel("Normalized RMSE (lower is better)")
ax.tick_params(axis="x",rotation=30)
fig.tight_layout()
fig.savefig(OUT/"validation_nrmse.png",dpi=180)
plt.show()
print("Next: notebook 13 performs fresh discovery for every new SAE checkpoint.")